# Numerical Stability Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Demonstrate floating point precision limits

In [ ]:
```python

print("=== Floating Point Precision ===")

print(f"0.1 + 0.2 = {0.1 + 0.2}")

print(f"0.1 + 0.2 == 0.3? {0.1 + 0.2 == 0.3}")

print(f"Difference: {(0.1 + 0.2) - 0.3:.2e}")

In [ ]:
```

### Step 2: Implement naive vs stable softmax

In [ ]:
```python

import math

def softmax_naive(logits):

    exps = [math.exp(z) for z in logits]

    total = sum(exps)

    return [e / total for e in exps]

def softmax_stable(logits):

    max_logit = max(logits)

    exps = [math.exp(z - max_logit) for z in logits]

    total = sum(exps)

    return [e / total for e in exps]

safe_logits = [2.0, 1.0, 0.1]

print(f"Naive:  {softmax_naive(safe_logits)}")

print(f"Stable: {softmax_stable(safe_logits)}")

dangerous_logits = [100.0, 101.0, 102.0]

print(f"Stable: {softmax_stable(dangerous_logits)}")

# softmax_naive(dangerous_logits) would return [nan, nan, nan]

In [ ]:
```

### Step 3: Implement stable log-sum-exp

In [ ]:
```python

def logsumexp_naive(values):

    return math.log(sum(math.exp(v) for v in values))

def logsumexp_stable(values):

    c = max(values)

    return c + math.log(sum(math.exp(v - c) for v in values))

safe = [1.0, 2.0, 3.0]

print(f"Naive:  {logsumexp_naive(safe):.6f}")

print(f"Stable: {logsumexp_stable(safe):.6f}")

large = [500.0, 501.0, 502.0]

print(f"Stable: {logsumexp_stable(large):.6f}")

# logsumexp_naive(large) returns inf

In [ ]:
```

### Step 4: Implement stable cross-entropy

In [ ]:
```python

def cross_entropy_naive(true_class, logits):

    probs = softmax_naive(logits)

    return -math.log(probs[true_class])

def cross_entropy_stable(true_class, logits):

    max_logit = max(logits)

    shifted = [z - max_logit for z in logits]

    log_sum_exp = math.log(sum(math.exp(s) for s in shifted))

    log_prob = shifted[true_class] - log_sum_exp

    return -log_prob

logits = [2.0, 5.0, 1.0]

true_class = 1

print(f"Naive:  {cross_entropy_naive(true_class, logits):.6f}")

print(f"Stable: {cross_entropy_stable(true_class, logits):.6f}")

In [ ]:
```

### Step 5: Gradient checking

In [ ]:
```python

def numerical_gradient(f, x, h=1e-5):

    grad = []

    for i in range(len(x)):

        x_plus = x[:]

        x_minus = x[:]

        x_plus[i] += h

        x_minus[i] -= h

        grad.append((f(x_plus) - f(x_minus)) / (2 * h))

    return grad

def check_gradient(analytical, numerical, tolerance=1e-5):

    for i, (a, n) in enumerate(zip(analytical, numerical)):

        denom = max(abs(a), abs(n), 1e-8)

        rel_error = abs(a - n) / denom

        status = "OK" if rel_error < tolerance else "FAIL"

        print(f"  param {i}: analytical={a:.8f} numerical={n:.8f} "

              f"rel_error={rel_error:.2e} [{status}]")

def f(params):

    x, y = params

    return x**2 + 3*x*y + y**3

def f_grad(params):

    x, y = params

    return [2*x + 3*y, 3*x + 3*y**2]

point = [2.0, 1.0]

analytical = f_grad(point)

numerical = numerical_gradient(f, point)

check_gradient(analytical, numerical)

In [ ]:
```

## Exercises

In [ ]:
1. **Catastrophic cancellation.** Compute the variance of [1000000.0, 1000001.0, 1000002.0] using the naive formula `E[x^2] - E[x]^2` in float32. Then compute it using Welford's online algorithm. Compare the errors against the true variance (0.6667).

2. **Precision hunt.** Find the smallest positive float32 value `x` such that `1.0 + x == 1.0` in Python. This is the machine epsilon. Verify it matches `numpy.finfo(numpy.float32).eps`.

3. **Log-sum-exp edge cases.** Test your `logsumexp_stable` function with: (a) all values equal, (b) one value much larger than the rest, (c) all values very negative (-1000). Verify it gives correct results where the naive version fails.

4. **Gradient checking a neural network layer.** Implement a single linear layer `y = Wx + b` and its analytical backward pass. Use `numerical_gradient` to verify correctness for a 3x2 weight matrix.

5. **Loss scaling experiment.** Simulate training with float16: create random gradients in the range [1e-9, 1e-3], convert to float16, and measure what fraction become zero. Then apply loss scaling (multiply by 1024), convert to float16, scale back, and measure the zero fraction again.